# Platypus GPU run

Runs one Platypus build on a Colab GPU and keeps the log, a manifest of exactly what ran, and a readout summary.

**Secrets:** keys live only in Colab Secrets (the key icon on the left). This notebook reads them with `userdata.get` and never prints them. Nothing here needs a key unless the dataset ever requires a Hugging Face sign-in.

Copyright Christian Bird. Apache 2.0.

## 1. Choose the run

In [ ]:
BUILD   = 'platypus'           # the package (python -m platypus); or a single-file build
PRESET  = 'configs/presets/cg27a_distillbr_pass1.env'  # switches (cg27a.env: the full build; cg27a_pass1.env: pass 1 only; cg27a_ablate.env: every private part off)
SEED    = 4
CONTROLS = []          # e.g. ['configs/controls/random_dreams.env']
REPO    = 'https://github.com/chrisbirduk/platypus.git'
BRANCH  = 'cg27a-distill'
USE_DRIVE = True       # keep the Pile cache and the run folders on Drive
DRIVE_DIR = '/content/drive/MyDrive/platypus'

## 2. Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type'
print('torch', torch.__version__, 'cuda', torch.version.cuda)

## 3. Drive, code and dependencies

The code is the `platypus` package in the repository. To run a single-file build instead (an older build, or the output of `scripts/export_single_file.py`), upload it to `/content/platypus/` with the file panel and set `BUILD` to its name.

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.isdir('/content/platypus/.git'):
    !git clone -q --branch {BRANCH} {REPO} /content/platypus
else:   # a second run in the same runtime: bring the clone up to date with the branch (it used to keep the old code)
    !git -C /content/platypus fetch -q origin {BRANCH} && git -C /content/platypus checkout -q {BRANCH} && git -C /content/platypus reset -q --hard origin/{BRANCH}
%cd /content/platypus
!git log -1 --oneline
!pip -q install datasets

## 4. Optional secrets

Only if a Hugging Face sign-in is ever needed. The value goes into the environment of this session only.

In [ ]:
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets (not shown)')
except Exception:
    print('no HF_TOKEN in Colab Secrets: fine for the public dataset')

## 5. Run

Stop early on a clear failure: a Traceback, routing marked WRONG, BPC far above the reference, or the brake rolling back every sleep.

In [ ]:
os.environ['PILE_CACHE'] = f'{DRIVE_DIR}/pile_cache' if USE_DRIVE else 'pile_cache'
os.environ['RUNS_DIR']   = f'{DRIVE_DIR}/runs' if USE_DRIVE else 'runs'
ctl = ' '.join(CONTROLS)
!bash scripts/run_gpu.sh {BUILD} {PRESET} {SEED} {ctl}

## 6. Summary of the latest run

In [ ]:
import glob
run = max(glob.glob(os.environ['RUNS_DIR'] + '/*'), key=os.path.getmtime)   # the newest run (folder names do not sort by date)
print(run)
print(open(run + '/readouts.md').read())

## 7. Before sharing the log

`run_gpu.sh` already scanned the run folder. Re-run the scan on anything else you send.

In [ ]:
!python3 scripts/scan_secrets.py {run}